[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/spinning_wheel/spinning_wheel.ipynb)

# A Spinning Wheel in the Plane and in Space

This example computes the angular momentum and the kinetic energy of a spinning wheel, and the torque and the power of a push at its rim: for a wheel in the plane, for the same wheel tilted in space, and each time for the wheel's mirror image as well. A first course in mechanics treats the plane and space separately. In the plane, the speed of turning, the angular momentum and the torque are signed numbers, counted positive counterclockwise; in space they are vectors along the axle, pointed by the right-hand rule. Here the same lines give all four quantities in both settings, and for the mirror image, which turns the other way.

The rate of turning is modelled as the plane the wheel turns in, an extrinsic bivector, and the places of the masses, counted from the hub, as intrinsic antivectors. A mass moves along `point.commutator(rate)`. Its angular momentum is its place joined with its mass-weighted velocity, and the torque of a push is the place it is applied at joined with the push, `handle & push`; both are intrinsic. The inertia is the map from rates to momenta, the kinetic energy is `(momentum & rate) / 2` and the power is `torque & rate`, each an intrinsic element paired with an extrinsic one. In the plane the angular momentum is a single number and in space it has three components, those a first course gives a vector along the axle; the expressions are the same, and only the algebra they are evaluated in differs. One reflection, `mirror >> x`, acts on every ingredient.

In the vector notation of mechanics, a mass's velocity is $\mathbf v = \boldsymbol\omega\times\mathbf r$, the angular momentum $\mathbf L = \sum_i m_i\,\mathbf r_i\times\mathbf v_i = I\boldsymbol\omega$, the kinetic energy $K = \tfrac12 I\omega^2$, the torque $\boldsymbol\tau = \mathbf r\times\mathbf F$ and the power $P = \boldsymbol\tau\cdot\boldsymbol\omega$. In the plane the cross products reduce to signed numbers, and a mirror $M$ takes a position to $M\mathbf r$ but each of the axial vectors $\boldsymbol\omega$, $\mathbf L$ and $\boldsymbol\tau$ to $\det(M)\,M$ times itself.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA2D, VGA3D
from examples.util.animation import show_animation
from examples.orientation.spinning_wheel import render

np.set_printoptions(precision=3, suppress=True)

In [ ]:
ga = VGA2D
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Point = ga.gatype.antivector()                # Antivector-
Rate = ga.gatype.bivector()                   # Bivector+

## Angular momentum, energy, torque and power in the plane

Three masses sit on a rim of unit radius and turn about the hub at a steady rate, and a push acts on the first of them along its motion. The inertia maps the wheel's rate of turning to its angular momentum; half the momentum paired with the rate is the kinetic energy, and the torque paired with the rate is the power the push delivers.

In [ ]:
masses = np.array([1, 1.5, 0.5])
# How far round the rim each mass sits from the first, in radians.
mass_angles = np.array([0, 2.1, 4.0])
# The wheel's speed of turning, in radians a unit of time.
speed = 3
push_strength = 1.5

In [ ]:
wheel_plane = mv.xy                                              # [] Rate+
# Each mass sits on the rim where the spoke comes to, turned in the wheel's plane by the mass's angle.
seats = (wheel_plane * (-mass_angles / 2)).exp()                 # [3] Rotor+
points = seats >> mv.x.dual()                                    # [3] Point-
rate = wheel_plane * speed                                       # [] Rate+
handle = points[0]                                               # [] Point-
# The push acts along the handle's motion.
push = handle.commutator(wheel_plane) * push_strength            # [] Point-

In [ ]:
# The inertia maps a rate of turning to the angular momentum: each mass's place joined with its velocity.
inertia = points.regressive(points.commutator(Rate) * masses).sum(axis=0)   # [] Momentum- <- Rate+
momentum = inertia(rate)                                         # [] Momentum-
energy = (momentum & rate) / 2                                   # [] Scalar+
# A push turns the wheel through the place it is applied at.
torque = handle & push                                           # [] Torque-
power = torque & rate                                            # [] Scalar+

print("Angular momentum", momentum.coefficient_table(), sep="\n")
print("Kinetic energy", energy.coefficient_table(), sep="\n")
print("Torque", torque.coefficient_table(), sep="\n")
print("Power", power.coefficient_table(), sep="\n")

## The mirror image

The mirror image of the wheel is the wheel built from the mirror images of its ingredients: the places of its masses, its rate of turning and the push. The same lines, on the two wheels at once, give the mirror image an angular momentum and a torque of the opposite sign, since it turns the other way, and the same energy and power.

In [ ]:
mirror = mv.x                                                    # [] Plane+
n_frames = 81
frame_duration_ms = 40
# The times of the frames, over one turn of the wheel.
times = np.linspace(0, 2 * np.pi / speed, n_frames, endpoint=False)

In [ ]:
# The mirror image of each ingredient, beside the ingredient.
wheel_points = stack([points, mirror >> points])                 # [2, 3] Point-
wheel_handles = stack([handle, mirror >> handle])                # [2] Point-
wheel_pushes = stack([push, mirror >> push])                     # [2] Point-
wheel_rates = stack([rate, mirror >> rate])                      # [2] Rate+

inertias = wheel_points.regressive(wheel_points.commutator(Rate) * masses).sum(axis=-1)   # [2] Momentum- <- Rate+
momenta = inertias(wheel_rates)                                  # [2] Momentum-
energies = (momenta & wheel_rates) / 2                           # [2] Scalar+
torques = wheel_handles & wheel_pushes                           # [2] Torque-
powers = torques & wheel_rates                                   # [2] Scalar+

print("Angular momentum of the mirror image", momenta[1].coefficient_table(), sep="\n")
print("Kinetic energy of the mirror image", energies[1].coefficient_table(), sep="\n")
print("Torque on the mirror image", torques[1].coefficient_table(), sep="\n")
print("Power delivered to the mirror image", powers[1].coefficient_table(), sep="\n")

In [ ]:
def turning(points, pushes, rates, times):
    """Yields turned mass positions, mass velocities, and handle pushes across `times`."""
    for time in times:
        turns = (rates * (-time / 2)).exp()                      # [2] Rotor+
        places = turns[:, None] >> points                        # [2, 3] Point-
        # A mass turning at a rate moves along its commutator with the rate.
        yield places, places.commutator(rates[:, None]), turns >> pushes


states = turning(wheel_points, wheel_pushes, wheel_rates, times)
show_animation(render.animate(wheel_points, wheel_rates, masses, states), "spinning_wheel_plane", frame_duration_ms)

The dashed line is the mirror, with the wheel on its left and the mirror image on its right. Colours identify matching masses and dot sizes show the masses; each coloured arrow is a mass's velocity, and the dark arrow is the push at the handle.

## The wheel tilted in space

The same wheel is set in space, in a plane tilted out of the plane xy by a turn in the plane yz, beside its image in the same mirror. The lines are those of the plane, evaluated in the algebra of space. The angular momentum and the torque now have three components each, and the energy and the power are the numbers found in the plane.

In [ ]:
ga = VGA3D
mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                # Antivector-
Rate = ga.gatype.bivector()                   # Bivector+

In [ ]:
# How far the wheel's plane is tilted out of the plane xy, by a turn in the plane yz, in radians.
tilt_angle = 0.9

In [ ]:
tilt = (mv.yz * (-tilt_angle / 2)).exp()                         # [] Rotor+
wheel_plane = tilt >> mv.xy                                      # [] Rate+
seats = (wheel_plane * (-mass_angles / 2)).exp()                 # [3] Rotor+
points = seats >> mv.x.dual()                                    # [3] Point-
rate = wheel_plane * speed                                       # [] Rate+
handle = points[0]                                               # [] Point-
push = handle.commutator(wheel_plane) * push_strength            # [] Point-
mirror = mv.x                                                    # [] Plane+

tilted_points = stack([points, mirror >> points])                # [2, 3] Point-
tilted_handles = stack([handle, mirror >> handle])               # [2] Point-
tilted_pushes = stack([push, mirror >> push])                    # [2] Point-
tilted_rates = stack([rate, mirror >> rate])                     # [2] Rate+

inertias = tilted_points.regressive(tilted_points.commutator(Rate) * masses).sum(axis=-1)   # [2] Momentum- <- Rate+
tilted_momenta = inertias(tilted_rates)                          # [2] Momentum-
tilted_energies = (tilted_momenta & tilted_rates) / 2            # [2] Scalar+
tilted_torques = tilted_handles & tilted_pushes                  # [2] Torque-
tilted_powers = tilted_torques & tilted_rates                    # [2] Scalar+

for wheel, label in enumerate(("the wheel", "the mirror image")):
    print(f"Angular momentum of {label}", tilted_momenta[wheel].coefficient_table(), sep="\n")
    print(f"Kinetic energy of {label}", tilted_energies[wheel].coefficient_table(), sep="\n")
    print(f"Torque on {label}", tilted_torques[wheel].coefficient_table(), sep="\n")
    print(f"Power delivered to {label}", tilted_powers[wheel].coefficient_table(), sep="\n")

states = turning(tilted_points, tilted_pushes, tilted_rates, times)
show_animation(render.animate(tilted_points, tilted_rates, masses, states), "spinning_wheel_space", frame_duration_ms)

The picture reads the wheel by the planes x and y, as the picture of the plane does, so the tilted rim shows as an ellipse; the mirror, colours and arrows are as before.

In [ ]:
# checks
# Kinetic energy equals half the mass-weighted squared speeds, and power equals the push dotted with the handle velocity.
velocities = wheel_points.commutator(wheel_rates[:, None])       # [2, 3] Point-
assert ((energies - (masses * velocities.dot(velocities)).sum(axis=-1) / 2).abs() < 1e-9).all()
assert ((powers - wheel_pushes.dot(velocities[:, 0])).abs() < 1e-9).all()
# In the plane the mirror image has the opposite angular momentum and identical kinetic energy and power.
assert (momenta[0] + momenta[1]).antinorm() < 1e-9
assert (energies[1] - energies[0]).abs() < 1e-9
assert (powers[1] - powers[0]).abs() < 1e-9
# Tilted in space, both wheels match the planar energy and power, and the mirrored wheel's momentum equals the mirrored momentum.
embed = mv.context.include_from_algebra
assert ((tilted_energies - embed(energies)).abs() < 1e-9).all()
assert ((tilted_powers - embed(powers)).abs() < 1e-9).all()
assert ((mirror >> tilted_momenta[0]) - tilted_momenta[1]).antinorm() < 1e-9